# Notebook 08: Significance

Primary tests now operate on run-level per-architecture data. transfer-gap significance per architecture, and retains DeLong/McNemar architecture contrasts. Holm-Bonferroni applied across all tests carrying an explicit p-value.


## 1. Setup and Load Predictions

In [1]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

import sys
from pathlib import Path

REPO_ON_DRIVE = "/content/drive/MyDrive/MSc AI DISSERTATION/crosspop-cxr-asymmetry"
for candidate in [REPO_ON_DRIVE, "..", "."]:
    p = Path(candidate).resolve()
    if (p / "config.py").exists():
        sys.path.insert(0, str(p))
        break

import importlib
import numpy as np
import pandas as pd

import config
from src import metrics as MET
from src import stats as ST
from src import results as RES

for mod in (config, MET, ST, RES):
    importlib.reload(mod)

config.ensure_output_dirs()


Mounted at /content/drive


In [2]:
ext_summary = pd.read_csv(config.RESULTS_DIR / "04_inference_summary_extended.csv")
meta, arrays = RES.load_predictions(config.RESULTS_DIR / "predictions")

consolidated_results = []


## 2. Transfer-gap significance, per architecture (does asymmetry exclude 0?)

In [3]:
gap_table = pd.read_csv(config.RESULTS_DIR / "05_transfer_gaps.csv")
print("--- 1. Transfer-gap asymmetry significance ---")
for _, r in gap_table.iterrows():
    is_reliable = not (np.isnan(r["asym_lo"]) or np.isnan(r["asym_hi"])) and (r["asym_lo"] > 0 or r["asym_hi"] < 0)
    consolidated_results.append({
        "test": f"Transfer-gap asymmetry ({r['arch']})",
        "method": "Run-level bootstrap (paired per-seed)",
        "estimate": r["asymmetry"], "ci_lo": r["asym_lo"], "ci_hi": r["asym_hi"],
        "p_value": np.nan, "significant": is_reliable,
    })
    print(f"{r['arch']:16s} asymmetry={r['asymmetry']:+.3f}  95% CI [{r['asym_lo']:+.3f}, {r['asym_hi']:+.3f}] "
          f"-> {'reliable' if is_reliable else 'not reliable'}")


--- 1. Transfer-gap asymmetry significance ---
mobilenet_v2     asymmetry=+0.233  95% CI [+0.125, +0.351] -> reliable
efficientnet_b0  asymmetry=+0.444  95% CI [+0.328, +0.530] -> reliable


## 3. Architecture contrasts within N2K (DeLong + McNemar)

In [4]:
print("\n--- 2. Architecture Contrasts (MobileNetV2 vs EfficientNet-B0, N2K) ---")

def extract_architecture_predictions(condition, arch_name):
    selection = meta[(meta.condition == condition) & (meta.arch == arch_name)].sort_values(
        [c for c in ["seed", "fold"] if c in meta.columns]
    )
    prob_list, y_ref = [], None
    for _, row in selection.iterrows():
        entry = arrays[row["stem_full"]]
        if y_ref is None:
            y_ref = entry["y_true"]
        prob_list.append(entry["y_prob"])
    return y_ref, np.mean(prob_list, axis=0)

y_true_n2k, prob_mobilenet = extract_architecture_predictions("N2K", "mobilenet_v2")
_, prob_efficientnet = extract_architecture_predictions("N2K", "efficientnet_b0")

# DeLong Test for AUROC Difference
auc_mob, auc_eff, p_val_delong = ST.delong_test(y_true_n2k, prob_mobilenet, prob_efficientnet)
print(f"DeLong (ensemble-averaged AUROC): MobileNetV2={auc_mob:.3f}, EfficientNet-B0={auc_eff:.3f} | p={p_val_delong:.4f}")
print("(Compare against run-level mean AUROC in 05_run_level_auroc.csv -- these differ by "
      "construction; see the note above section 2.)")

consolidated_results.append({
    "test": "Architecture contrast AUROC (N2K, ensemble-averaged)", "method": "DeLong",
    "estimate": round(auc_mob - auc_eff, 3), "ci_lo": np.nan, "ci_hi": np.nan,
    "p_value": round(p_val_delong, 4), "significant": p_val_delong < 0.05
})

# McNemar Test for Classification Accuracy Difference (at threshold 0.5)
pred_mobilenet = (prob_mobilenet >= 0.5).astype(int)
pred_efficientnet = (prob_efficientnet >= 0.5).astype(int)

b, c, p_val_mcnemar = ST.mcnemar_test(y_true_n2k, pred_mobilenet, pred_efficientnet)
print(f"McNemar (Accuracy, ensemble-averaged predictions): b={b}, c={c} | p={p_val_mcnemar:.4f}")

consolidated_results.append({
    "test": "Architecture contrast accuracy (N2K, ensemble-averaged)", "method": "McNemar",
    "estimate": np.nan, "ci_lo": np.nan, "ci_hi": np.nan,
    "p_value": round(p_val_mcnemar, 4), "significant": p_val_mcnemar < 0.05
})



--- 2. Architecture Contrasts (MobileNetV2 vs EfficientNet-B0, N2K) ---
DeLong (ensemble-averaged AUROC): MobileNetV2=0.738, EfficientNet-B0=0.889 | p=0.0000
(Compare against run-level mean AUROC in 05_run_level_auroc.csv -- these differ by construction; see the note above section 2.)
McNemar (Accuracy, ensemble-averaged predictions): b=54, c=116 | p=0.0000


## 4. Holm-Bonferroni correction and consolidated export

In [5]:
results_df = pd.DataFrame(consolidated_results)

# Apply Holm-Bonferroni step-down correction for tests with explicit p-values
p_value_mask = results_df["p_value"].notna()
if p_value_mask.any():
    p_vals = results_df.loc[p_value_mask, "p_value"]
    sorted_indices = p_vals.sort_values().index
    m_tests = len(p_vals)

    holm_p_adjusted = {}
    prev_adj_p = 0.0

    for rank, idx in enumerate(sorted_indices):
        raw_p = p_vals[idx]
        adj_p = min(1.0, (m_tests - rank) * raw_p)
        adj_p = max(adj_p, prev_adj_p)
        holm_p_adjusted[idx] = round(adj_p, 4)
        prev_adj_p = adj_p

    results_df["p_holm"] = results_df.index.map(lambda i: holm_p_adjusted.get(i, np.nan))
else:
    results_df["p_holm"] = np.nan

# Organize columns for final report
column_order = ["test", "method", "estimate", "ci_lo", "ci_hi", "p_value", "p_holm", "significant"]
final_table = results_df[column_order]

print("\n=========================================================================")
print("                  CONSOLIDATED STATISTICAL TEST SUMMARY                  ")
print("=========================================================================")
print(final_table.to_string(index=False))

csv_output_path = config.RESULTS_DIR / "08_significance.csv"
final_table.to_csv(csv_output_path, index=False)
print(f"\nSaved to {csv_output_path.name}")



                  CONSOLIDATED STATISTICAL TEST SUMMARY                  
                                                   test                                method  estimate  ci_lo  ci_hi  p_value  p_holm  significant
                  Transfer-gap asymmetry (mobilenet_v2) Run-level bootstrap (paired per-seed)     0.233  0.125  0.351      NaN     NaN         True
               Transfer-gap asymmetry (efficientnet_b0) Run-level bootstrap (paired per-seed)     0.444  0.328  0.530      NaN     NaN         True
   Architecture contrast AUROC (N2K, ensemble-averaged)                                DeLong    -0.151    NaN    NaN      0.0     0.0         True
Architecture contrast accuracy (N2K, ensemble-averaged)                               McNemar       NaN    NaN    NaN      0.0     0.0         True

Saved to 08_significance.csv


**Next:** notebook 09 (ood-musa).

Exploratory Out-of-Distribution response probing over-confidence on unseen Musa pathologies (TB / COVID-19).